#### Copyright 2026 Google LLC

In [ ]:
# @title Licensed under the Apache License, version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

### **NOTE**: This Jupyter Notebook is intended for use on your local system with a Jupyter Notebook interface

# Overview

### Introduction

The Google Tensor SDK is used to compile LiteRT models for Pixel devices, for enhanced ML inference performance.

*This Jupyter notebook is based on the more general [LiteRT AOT Compilation Tutorial Colab]:
<a target="_blank" href="https://colab.research.google.com/github/google-ai-edge/litert-samples/blob/main/compiled_model_api/colab/LiteRT_AOT_Compilation_Tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>.*

### Objective

In this jupyter notebook, you will learn how to use the **LiteRT AOT** (ahead of time) compiler to compile a [selfie segmentation model](https://ai.google.dev/edge/mediapipe/solutions/vision/image_segmenter#selfie-model) from a TFLite model, to LiteRT models that is optimized and compiled for on-device EdgeTPUs.

This notebook will also walk you through the steps for preparing the models with [Play for On-device AI]( https://developer.android.com/google/play/on-device-ai) (**PODAI**).

PODAI delivers custom models for on-device AI features more efficiently. It simplifies the process for launching, targeting, versioning, and downloading your AI models. When combined with LiteRT EdgeTPU AOT compilation, it lets developers deliver compiled ML models for various devices without the need to know what EdgeTPUs the end users' phone is equiped with.

### Models used

The models we are using are originally published on MediaPipe [Image segmentation guide](https://ai.google.dev/edge/mediapipe/solutions/vision/image_segmenter). These are the two models used in this colabs:

* [`SelfieMulticlass`](https://ai.google.dev/edge/mediapipe/solutions/vision/image_segmenter#multiclass-model): A LiteRT model that takes an image of a person, locates areas for different areas such as hair, skin, and clothing, and outputs an image segmentation map for these items.


# Prerequisite: Install the required packages

Start by installing the required packages, including the ai-edge-litert which contains the EdgeTPU AOT compiler, and other libraries you'll use for model conversion.

#### **Note** [only for colab users]: Uninstall certain packages

On Google Colab, before any module installation, we need to uninstall the tensorflow that comes with the colab runtime by default.

Note: You may encounter warnings during the *uninstall* process regarding packages not being present. These can be safely ignored.

Uninstall command:

`pip uninstall -y tensorflow ai-edge-litert`

### Install all the libraries

Note: The `pip install` commands might show an error due to a version mismatch. This is expected, please proceed.

In [ ]:
%env GOOGLE_TENSOR_SDK_BETA=/path/to/downloaded/litert_plugin_compiler.tar.gz
!ls $GOOGLE_TENSOR_SDK_BETA

!pip install ai-edge-litert-sdk-google-tensor==2.1.5

!pip install ai-edge-litert-nightly==2.2.0.dev20260518 matplotlib huggingface-hub

# Main Execution Workflow

Import the required packages.

In [ ]:
import os
import shutil

from ai_edge_litert.aot import aot_compile as aot_lib
from ai_edge_litert.aot.ai_pack import export_lib as ai_pack_export
from ai_edge_litert.aot.vendors.google_tensor import target as gt_target
import huggingface_hub
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests

## Compile a LiteRT model

This section covers advanced usages like compiling a LiteRT (TFLite) model directly.


### EdgeTPU Compilation from TFLite model

This step requires a TFLite model. If you do not have a TFLite model, convert your model into the TFLite format.

#### Getting the TFLite Model

We will use [MediaPipe MultiClass Segmentation](https://storage.googleapis.com/mediapipe-assets/Model%20Card%20Multiclass%20Segmentation.pdf) model for this use case.

The TFLite model is available from [MediaPipe Image segmentation](https://ai.google.dev/edge/mediapipe/solutions/vision/image_segmenter#multiclass-model) page.


In [ ]:
work_dir = '.'

model_url = 'https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite'
tflite_model_path = os.path.join(work_dir, 'selfie_multiclass_256x256.tflite')

model_content = requests.get(model_url)

with open(tflite_model_path, 'wb') as fout:
  fout.write(model_content.content)

#### Quickly verify the TfLite model using LiteRT Python API

In the following example, we will show both mask image and blended result.

In [ ]:
# Downloading Testing image

test_image = huggingface_hub.hf_hub_download(
    repo_id="litert-community/MediaPipe-Selfie-Segmentation",
    filename="test_img.png",
)
pil_image = Image.open(test_image).convert("RGB").resize((256, 256))

In [ ]:
from ai_edge_litert.compiled_model import CompiledModel

SEGMENT_COLORS = [
    (0, 0, 0),
    (255, 0, 0),
    (0, 255, 0),
    (0, 0, 255),
    (255, 255, 0),
    (255, 0, 255),
]
INPUT_SIZE = (256, 256)
NUM_CLASSES = 6

# Load the model and image
model = CompiledModel.from_file(tflite_model_path)
original_image = np.array(Image.open(test_image).convert('RGB'))
img_array = np.array(pil_image).astype(np.float32)

# Normalize the image
normalized = (img_array - 127.5) / 127.5
normalized = np.ascontiguousarray(normalized, dtype=np.float32)

# Run inference
sig_idx = 0
input_buffers = model.create_input_buffers(sig_idx)
output_buffers = model.create_output_buffers(sig_idx)
input_data = normalized.reshape(-1)
input_buffers[0].write(input_data)
model.run_by_index(sig_idx, input_buffers, output_buffers)

# Get output data
height, width = INPUT_SIZE
output_size = height * width * NUM_CLASSES
output_data = output_buffers[0].read(output_size, np.float32)
output_data = output_data.reshape(height, width, NUM_CLASSES)
mask = np.argmax(output_data, axis=2).astype(np.uint8)

# Create colored mask
colored_mask = np.zeros((height, width, 3), dtype=np.uint8)
for label_idx in range(NUM_CLASSES):
  class_mask = mask == label_idx
  color = SEGMENT_COLORS[label_idx]
  colored_mask[class_mask] = color

# Blend with original image
# Resize colored mask to match original image if necessary
if original_image.shape[:2] != colored_mask.shape[:2]:
  colored_mask_pil = Image.fromarray(colored_mask)
  colored_mask_pil = colored_mask_pil.resize(
      (original_image.shape[1], original_image.shape[0])
  )
  colored_mask = np.array(colored_mask_pil)

# Blend images with alpha 0.5
alpha = 0.5
blended_image = (original_image * (1 - alpha) + colored_mask * alpha).astype(
    np.uint8
)

# Display them
fig, axes = plt.subplots(1, 3, figsize=(9, 3))

for idx, (title, image) in enumerate([
    ('Original Image', original_image),
    ('Colored Mask', colored_mask),
    ('Blended Image', blended_image),
]):
  axes[idx].imshow(image)
  axes[idx].set_title(title)
  axes[idx].axis('off')

plt.tight_layout()
plt.show()

#### Convert to LiteRT model, with EdgeTPU AOT compilation.

We use the APIs from `ai_edge_litert.aot` to compile the model.

In [ ]:
compiled_models = aot_lib.aot_compile(tflite_model_path, keep_going=True)

# This variable will be used later to create the AI Pack.
all_google_tensor_compiled_models = compiled_models

In [ ]:
# @title Print Compilation Report

print(all_google_tensor_compiled_models.compilation_report())

In [ ]:
# Saving compiled models to disk. This saves all the compiled models, and a CPU
# fallback model.

all_google_tensor_compiled_models.export(
    work_dir, model_name='selfie_segmentation'
)

Once the compilation finishes, we can use `model.export` method to export all models to disk.

By default, the models will be stored in a flat structure in output directory, with each model name suffixed with the backend id.

For example:

| Model File Name                             | Backend  | SoC       | Note               |
|---------------------------------------------|----------|-----------|--------------------|
| selfie_segmentation_fallback.tflite         | CPU/GPU  | N/A       | N/ A               |
| selfie_segmentation_Google_Tensor_G3.tflite | Google   | Tensor_G3 | Google Tensor G3   |
| selfie_segmentation_Google_Tensor_G4.tflite | Google   | Tensor_G4 | Google Tensor G4   |
| selfie_segmentation_Google_Tensor_G5.tflite | Google   | Tensor_G5 | Google Tensor G5   |


## Export and validate on CPU

Once the compilation is complete, verify the TFLite model on the CPU. Do this using the "fallback model" generated during compilation.

In [ ]:
# Run LiteRT with test image
from ai_edge_litert.compiled_model import CompiledModel

# Normalize the image to [-1, 1]
img_array = np.array(pil_image, dtype=np.float32)
normalized = (img_array - 127.5) / 127.5
numpy_array = np.ascontiguousarray(normalized)[None, ...]

cpu_model_path = os.path.join(work_dir, "selfie_segmentation_fallback.tflite")
cm_model = CompiledModel.from_file(cpu_model_path)
sig_idx = 0
input_buffers = cm_model.create_input_buffers(sig_idx)
output_buffers = cm_model.create_output_buffers(sig_idx)
input_buffers[0].write(numpy_array)
cm_model.run_by_index(sig_idx, input_buffers, output_buffers)

# Read the 6-channel output and apply argmax
output_data = output_buffers[0].read(256 * 256 * 6, np.float32).reshape((256, 256, 6))
mask = np.argmax(output_data, axis=2).astype(np.uint8)

# Create a colored mask using the previously defined SEGMENT_COLORS
colored_mask = np.zeros((256, 256, 3), dtype=np.uint8)
for label_idx in range(6):
  class_mask = mask == label_idx
  color = SEGMENT_COLORS[label_idx]
  colored_mask[class_mask] = color

mask_image = Image.fromarray(colored_mask)


In [ ]:
# Show output results

fig, axes = plt.subplots(1, 2, figsize=(9, 3))

for idx, (title, image) in enumerate([
    ('Test Image', pil_image),
    ('TFLite Mask Image', mask_image),
]):
  axes[idx].imshow(image)
  axes[idx].set_title(title)
  axes[idx].axis('off')

plt.tight_layout()
plt.show()

## Export Models for Google Play On-Device AI (PODAI)

With your models verified, the next essential step is preparing them for deployment. This section details how to package your compiled models for upload to Google Play, enabling delivery to user devices through the On-Demand AI (PODAI) framework.

The AiEdgeLiteRT AOT (Ahead-of-Time) module provides `ai_pack` utilities specifically for this purpose. These utilities create an **AI Pack**, which is a crucial data asset. An AI Pack bundles your compiled models with device-targeting configurations, ensuring that the correct models and assets are delivered to the appropriate user devices. This is particularly vital for NPU (Neural Processing Unit) compilations, as it guarantees that models optimized for a specific System-on-Chip (SoC) reach only the devices equipped with that SoC.

In [ ]:
# Configuring the AI Pack
os.makedirs('selfie_multiclass', exist_ok=True)
ai_pack_dir = os.path.join(work_dir, 'ai_pack')
ai_pack_name = 'selfie_segmentation'
litert_model_name = 'segmentation_model'

# Clean up`
shutil.rmtree(ai_pack_dir, ignore_errors=True)

# Export
ai_pack_export.export(
    all_google_tensor_compiled_models, ai_pack_dir, ai_pack_name, litert_model_name
)

In [ ]:
# @title Inspecting AI Pack source

def list_files(startpath):
  """Function to print out the tree structure of a directory."""
  for root, dirs, files in os.walk(startpath):
    level = root.replace(startpath, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print('{}{}/'.format(indent, os.path.basename(root)))
    subindent = ' ' * 4 * (level + 1)
    for f in files:
      print('{}{}'.format(subindent, f))

In [ ]:
list_files(ai_pack_dir)

### Result

And now the models are ready for comsuption by PODAI!

Now we will move on to Android Studio for the following steps, please refer to [LiteRT image segmentation samples](https://github.com/google-ai-edge/litert-samples/tree/main/v2/image_segmentation) for details.

But if you are curious on the contents of AI Pack, we can take a look into the directory.

# Advanced Options

## Compilation for specific Google Tensor device or EdgeTPU

By default, LiteRT AOT compilation compiles to all registered backends. But for local developement, you might only want to compile for specific devices, say the development phones by hand. This is achievable by providing the compilation targets explicitly.

The following example will compile to Google Tensor G4 and Google Tensor G5.

In [ ]:
# Specifying the compilation target

tensor_g5_target = gt_target.Target(gt_target.SocModel.TENSOR_G5)

In [ ]:
# Compile from the TFLite model for a specific target

compiled_models = aot_lib.aot_compile(
    tflite_model_path,
    target=[tensor_g5_target],
    keep_going=False,  # We want to error out when there's failure.
)

In [ ]:
print(compiled_models.compilation_report())

## Compilation flags for Google Tensor

Compilation process can be customized through compilation flags.


#### When compiling a TFLite model

```
compiled_models = aot_lib.aot_compile(tflite_model_path, target=[tensor_g5_target], flag_name1=value1, flag_name2=value2, ...)
```



The following example will show using flags for Google Tensor G5.

In [ ]:
# Compiling with a compilation flag
compiled_models = aot_lib.aot_compile(
    tflite_model_path,
    target=[tensor_g5_target],
    keep_going=False,
    google_tensor_truncation_type="half"
)